In [ ]:
# Cell 0
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 1 — Clone / Update Repo
repo_url = "https://github.com/siddheshkadane01/Site-Reliability-Server/"
repo_dir = repo_url.rstrip("/").split("/")[-1]

import os
if os.path.exists(repo_dir):
    !git -C {repo_dir} pull origin sriram
else:
    !git clone -b sriram {repo_url}

%cd {repo_dir}

In [ ]:
# Cell 2 - All installs in one cell, correct order:
%pip install -r requirements.txt
%pip install -r requirements_rl.txt

# Pin TRL (important for GRPO stability)
%pip install trl==0.18.2 --quiet

!pip show trl | grep Version

In [ ]:
# Cell 3 - Start FastAPI server with auto-restart monitor
import subprocess, time, requests, threading, os

def start_server():
    log_file = open("server.log", "a")
    process = subprocess.Popen(
        ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860",
         "--workers", "1", "--timeout-keep-alive", "300"],
        stdout=log_file, stderr=log_file
    )
    return process

def monitor_and_restart_server():
    global server_process
    while True:
        try:
            res = requests.get("http://localhost:7860/health", timeout=5)
            if res.status_code != 200:
                raise Exception("Unhealthy")
        except:
            print("⚠️ Server down — restarting...")
            try:
                server_process.kill()
            except:
                pass
            time.sleep(3)
            server_process = start_server()
            time.sleep(10)
            print("✅ Server restarted")
        time.sleep(15)  # check every 15 seconds

# Kill any existing server on port 7860
os.system("fuser -k 7860/tcp 2>/dev/null || true")
time.sleep(2)

# Start server
server_process = start_server()
time.sleep(12)

# Health check
try:
    res = requests.get("http://localhost:7860/health", timeout=5)
    print("✅ Server running" if res.status_code == 200 
          else f"❌ Unexpected status: {res.status_code}")
except Exception as e:
    print("❌ Server failed:", e)
    print(open("server.log").read())

# Start monitor thread (daemon=True means it dies when Colab session ends)
monitor_thread = threading.Thread(target=monitor_and_restart_server, daemon=True)
monitor_thread.start()
print("✅ Server monitor running — will auto-restart if server crashes")
print(open("server.log").read()[:300])

In [ ]:
# Diagonsis Cell
import requests

session = requests.Session()

# Test reset
r = session.post("http://localhost:7860/reset", 
                 json={"task_id": "enterprise"}, timeout=10)
print("Reset status:", r.status_code)
obs = r.json()
print("Obs keys:", list(obs.keys()) if isinstance(obs, dict) else obs)

# Test a step
r2 = session.post("http://localhost:7860/step",
                  json={"action_type": "CHECK_LOGS", 
                        "target_service": "api-gateway"}, timeout=10)
print("Step status:", r2.status_code)
result = r2.json()
print("Reward:", result.get("reward"))
print("Done:", result.get("done"))

In [ ]:
# Cell 4 - Wandb login using secret:
import wandb, os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

wandb.login()

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

print("✅ HuggingFace login successful")

In [ ]:
# Cell 5 - FINAL TRAINING RUN
import os
os.environ["WANDB_PROJECT"] = "openenv-enterprise-grpo"
os.environ["WANDB_RUN_NAME"] = "grpo-qwen3b-FINAL-v3"
os.environ["WANDB_MODE"] = "online"

!python train_grpo.py \
    --epochs 10 \
    --num_generations 6 \
    --per_device_train_batch_size 6 \
    --dataset_size 64 \
    --temperature 1.0 \
    --top_p 0.92 \
    --lora_dropout 0.05 \
    --learning_rate 5e-6 \
    --env_url http://localhost:7860 \
    --model_name Qwen/Qwen2.5-Coder-3B-Instruct \
    --max_new_tokens 32 \
    --max_prompt_length 512 \
    --output_dir ./artifacts/grpo-qwen3b-FINAL-v3